# AP20 - Pinecone in Production

### Master Pinecone vector database for production RAG systems

**Duration:** 2.5 hours

**What you'll learn:**
- Pinecone architecture and concepts
- Creating and managing indexes
- Upsert, query, and delete operations
- Metadata filtering
- Namespaces for multi-tenancy
- Hybrid search (dense + sparse)
- Performance optimization
- Cost management
- Production best practices

**Prerequisites:**
- Completed AP19 - Vector Database Fundamentals
- Understanding of vector embeddings
- Basic Python

---

## 🔧 Setup

In [ ]:
# Install dependencies
# !pip install -qU pinecone-client openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
import time

load_dotenv()

# Initialize Pinecone
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

print("✅ Pinecone initialized!")
print(f"   API Key: {os.getenv('PINECONE_API_KEY')[:10]}...")

---

## 1. Pinecone Architecture

### Key Concepts

```
Pinecone
  └── Index (e.g., "products-search")
      ├── Namespace 1 (e.g., "customer-a")
      │   └── Vectors with metadata
      └── Namespace 2 (e.g., "customer-b")
          └── Vectors with metadata
```

**Components:**
- **Index**: Container for vectors (like a database)
- **Namespace**: Partition within index (like schemas/tables)
- **Vector**: Embedding + ID + Metadata
- **Metadata**: Key-value pairs for filtering

### Serverless vs Pod-based

| Feature | Serverless | Pod-based |
|---------|-----------|----------|
| **Cost** | Pay per use | Fixed cost |
| **Scaling** | Automatic | Manual |
| **Startup** | Instant | Minutes |
| **Best For** | Variable load | Predictable load |
| **Control** | Less | More |

---

## 2. Creating Your First Index

In [ ]:
# Index configuration
index_name = "quickstart-index"
dimension = 1536  # OpenAI ada-002 dimension
metric = "cosine"  # cosine, euclidean, or dotproduct

# Check if index exists
existing_indexes = [idx.name for idx in pc.list_indexes()]

if index_name not in existing_indexes:
    print(f"Creating index '{index_name}'...")
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric=metric,
        spec=ServerlessSpec(
            cloud='aws',
            region='us-east-1'
        )
    )
    print("✅ Index created!")
    
    # Wait for index to be ready
    while not pc.describe_index(index_name).status['ready']:
        print("Waiting for index to be ready...")
        time.sleep(1)
else:
    print(f"✅ Index '{index_name}' already exists")

# Connect to index
index = pc.Index(index_name)

# Get index stats
stats = index.describe_index_stats()
print(f"\nIndex stats:")
print(f"  Dimension: {stats['dimension']}")
print(f"  Total vectors: {stats['total_vector_count']}")
print(f"  Namespaces: {list(stats.get('namespaces', {}).keys())}")

---

## 3. Upsert Operations - Adding Data

In [ ]:
from openai import OpenAI
import numpy as np

client = OpenAI()

def get_embedding(text: str) -> list:
    """Generate embedding using OpenAI"""
    response = client.embeddings.create(
        model="text-embedding-ada-002",
        input=text
    )
    return response.data[0].embedding

# Sample data
documents = [
    {
        "id": "doc1",
        "text": "Python is a versatile programming language for AI and data science.",
        "category": "programming",
        "level": "beginner",
        "views": 1000
    },
    {
        "id": "doc2",
        "text": "Machine learning algorithms learn patterns from data automatically.",
        "category": "ai",
        "level": "intermediate",
        "views": 2500
    },
    {
        "id": "doc3",
        "text": "Neural networks are inspired by biological neurons in the brain.",
        "category": "ai",
        "level": "advanced",
        "views": 1500
    }
]

# Prepare vectors for upsert
vectors_to_upsert = []

for doc in documents:
    # Generate embedding
    embedding = get_embedding(doc["text"])
    
    # Prepare vector with metadata
    vectors_to_upsert.append({
        "id": doc["id"],
        "values": embedding,
        "metadata": {
            "text": doc["text"],
            "category": doc["category"],
            "level": doc["level"],
            "views": doc["views"]
        }
    })
    print(f"Processed: {doc['id']}")

# Upsert to Pinecone
index.upsert(vectors=vectors_to_upsert, namespace="demo")

print(f"\n✅ Upserted {len(vectors_to_upsert)} vectors to namespace 'demo'")

# Verify
time.sleep(1)  # Wait for propagation
stats = index.describe_index_stats()
print(f"Total vectors: {stats['total_vector_count']}")

### Batch Upsert for Large Datasets

In [ ]:
def batch_upsert(documents, batch_size=100, namespace="default"):
    """Upsert documents in batches"""
    total = len(documents)
    
    for i in range(0, total, batch_size):
        batch = documents[i:i+batch_size]
        
        # Prepare batch
        vectors = [
            {
                "id": doc["id"],
                "values": get_embedding(doc["text"]),
                "metadata": {k: v for k, v in doc.items() if k not in ["id", "text"]}
            }
            for doc in batch
        ]
        
        # Upsert
        index.upsert(vectors=vectors, namespace=namespace)
        
        print(f"Upserted batch {i//batch_size + 1}: {len(vectors)} vectors")
        time.sleep(0.1)  # Rate limiting
    
    print(f"✅ Total upserted: {total} vectors")

# Example usage
# batch_upsert(large_document_list, batch_size=100)

---

## 4. Query Operations - Semantic Search

In [ ]:
def search(query: str, top_k=3, namespace="demo"):
    """Semantic search in Pinecone"""
    # Generate query embedding
    query_embedding = get_embedding(query)
    
    # Search
    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        namespace=namespace,
        include_metadata=True
    )
    
    return results

# Test search
query = "What is artificial intelligence?"
results = search(query, top_k=3)

print(f"Query: {query}\n")
print("Results:")
print("=" * 70)

for i, match in enumerate(results['matches']):
    print(f"\n{i+1}. ID: {match['id']}")
    print(f"   Score: {match['score']:.4f}")
    print(f"   Text: {match['metadata']['text']}")
    print(f"   Category: {match['metadata']['category']}")
    print(f"   Level: {match['metadata']['level']}")

---

## 5. Metadata Filtering

In [ ]:
def search_with_filter(query: str, filter_dict: dict, top_k=3, namespace="demo"):
    """Search with metadata filters"""
    query_embedding = get_embedding(query)
    
    results = index.query(
        vector=query_embedding,
        filter=filter_dict,
        top_k=top_k,
        namespace=namespace,
        include_metadata=True
    )
    
    return results

# Example 1: Filter by category
print("Filter 1: Only 'ai' category")
print("=" * 70)
results = search_with_filter(
    query="learning",
    filter_dict={"category": {"$eq": "ai"}}
)

for match in results['matches']:
    print(f"- {match['metadata']['text'][:50]}... ({match['metadata']['category']})")

# Example 2: Filter by multiple conditions
print("\nFilter 2: AI category AND intermediate+ level")
print("=" * 70)
results = search_with_filter(
    query="algorithms",
    filter_dict={
        "$and": [
            {"category": {"$eq": "ai"}},
            {"level": {"$in": ["intermediate", "advanced"]}}
        ]
    }
)

for match in results['matches']:
    print(f"- {match['metadata']['text'][:50]}... ({match['metadata']['level']})")

# Example 3: Numeric filter
print("\nFilter 3: More than 1200 views")
print("=" * 70)
results = search_with_filter(
    query="programming",
    filter_dict={"views": {"$gt": 1200}}
)

for match in results['matches']:
    print(f"- {match['metadata']['text'][:50]}... (views: {match['metadata']['views']})")

### Filter Operators

| Operator | Description | Example |
|----------|-------------|-------|
| `$eq` | Equals | `{"category": {"$eq": "ai"}}` |
| `$ne` | Not equals | `{"status": {"$ne": "draft"}}` |
| `$gt` | Greater than | `{"price": {"$gt": 100}}` |
| `$gte` | Greater than or equal | `{"rating": {"$gte": 4.0}}` |
| `$lt` | Less than | `{"age": {"$lt": 18}}` |
| `$lte` | Less than or equal | `{"stock": {"$lte": 10}}` |
| `$in` | In list | `{"color": {"$in": ["red", "blue"]}}` |
| `$nin` | Not in list | `{"tag": {"$nin": ["spam"]}}` |
| `$and` | Logical AND | `{"$and": [{...}, {...}]}` |
| `$or` | Logical OR | `{"$or": [{...}, {...}]}` |

---

## 6. Namespaces for Multi-Tenancy

In [ ]:
# Namespaces allow data isolation within the same index

# Company A's data
company_a_docs = [
    {"id": "a1", "text": "Company A product documentation for widget A."},
    {"id": "a2", "text": "Company A training materials for employees."}
]

# Company B's data
company_b_docs = [
    {"id": "b1", "text": "Company B internal knowledge base article."},
    {"id": "b2", "text": "Company B customer support FAQ."}
]

# Upsert to different namespaces
for doc in company_a_docs:
    vector = {
        "id": doc["id"],
        "values": get_embedding(doc["text"]),
        "metadata": {"text": doc["text"]}
    }
    index.upsert(vectors=[vector], namespace="company-a")

for doc in company_b_docs:
    vector = {
        "id": doc["id"],
        "values": get_embedding(doc["text"]),
        "metadata": {"text": doc["text"]}
    }
    index.upsert(vectors=[vector], namespace="company-b")

print("✅ Data upserted to separate namespaces")

# Query each namespace separately
query = "documentation"

print(f"\nQuery: {query}")
print("\nCompany A namespace:")
results_a = search(query, namespace="company-a", top_k=2)
for match in results_a['matches']:
    print(f"  - {match['metadata']['text'][:50]}...")

print("\nCompany B namespace:")
results_b = search(query, namespace="company-b", top_k=2)
for match in results_b['matches']:
    print(f"  - {match['metadata']['text'][:50]}...")

print("\n💡 Namespaces provide data isolation!")

---

## 7. Hybrid Search (Dense + Sparse)

In [ ]:
# Hybrid search combines:
# 1. Dense vectors (semantic similarity)
# 2. Sparse vectors (keyword matching - BM25)

def hybrid_search(query: str, alpha=0.5, top_k=3, namespace="demo"):
    """
    Hybrid search combining dense and sparse vectors
    
    alpha: weight for dense vectors (0-1)
           0 = pure keyword search
           1 = pure semantic search
           0.5 = balanced
    """
    # Get dense embedding
    dense_vector = get_embedding(query)
    
    # For hybrid search with Pinecone, you'd use:
    # 1. Pinecone's hybrid search feature (if available)
    # 2. Or combine results from separate queries
    
    # Simple approach: Semantic search with metadata filtering
    # Extract keywords from query
    keywords = query.lower().split()
    
    # Search semantically
    results = index.query(
        vector=dense_vector,
        top_k=top_k * 2,  # Get more, then rerank
        namespace=namespace,
        include_metadata=True
    )
    
    # Rerank based on keyword presence
    scored_matches = []
    for match in results['matches']:
        text = match['metadata']['text'].lower()
        
        # Keyword score
        keyword_score = sum(1 for kw in keywords if kw in text) / len(keywords)
        
        # Combined score
        hybrid_score = (alpha * match['score'] + (1-alpha) * keyword_score)
        
        match['hybrid_score'] = hybrid_score
        scored_matches.append(match)
    
    # Sort by hybrid score
    scored_matches.sort(key=lambda x: x['hybrid_score'], reverse=True)
    
    return scored_matches[:top_k]

# Test hybrid search
query = "machine learning algorithms"

print(f"Query: {query}\n")

print("Pure Semantic (alpha=1.0):")
results = hybrid_search(query, alpha=1.0)
for i, match in enumerate(results):
    print(f"{i+1}. {match['metadata']['text'][:50]}... (score: {match['hybrid_score']:.3f})")

print("\nBalanced (alpha=0.5):")
results = hybrid_search(query, alpha=0.5)
for i, match in enumerate(results):
    print(f"{i+1}. {match['metadata']['text'][:50]}... (score: {match['hybrid_score']:.3f})")

print("\n💡 Hybrid search combines semantic + keyword matching!")

---

## 8. Performance Optimization

In [ ]:
# 1. Batch operations
def efficient_upsert(documents, batch_size=100):
    """Efficient batch upsert"""
    # Group into batches
    for i in range(0, len(documents), batch_size):
        batch = documents[i:i+batch_size]
        
        # Parallel embedding generation (use asyncio in production)
        vectors = [
            {
                "id": doc["id"],
                "values": get_embedding(doc["text"]),
                "metadata": doc["metadata"]
            }
            for doc in batch
        ]
        
        index.upsert(vectors=vectors)

# 2. Fetch operations
def fetch_vectors(ids, namespace="demo"):
    """Fetch vectors by ID (no embedding needed)"""
    response = index.fetch(ids=ids, namespace=namespace)
    return response['vectors']

# Test fetch
vectors = fetch_vectors(["doc1", "doc2"])
print(f"Fetched {len(vectors)} vectors by ID")

# 3. Delete operations
def delete_by_filter(filter_dict, namespace="demo"):
    """Delete vectors matching filter"""
    index.delete(filter=filter_dict, namespace=namespace)

# 4. Update operations
def update_metadata(id, new_metadata, namespace="demo"):
    """Update vector metadata without re-embedding"""
    index.update(
        id=id,
        set_metadata=new_metadata,
        namespace=namespace
    )

# Example: Update views count
update_metadata("doc1", {"views": 2000}, namespace="demo")
print("✅ Metadata updated without re-embedding!")

### Performance Best Practices

1. **Batch Operations**: Always upsert/query in batches (100-1000 vectors)
2. **Metadata Strategy**: Only store what you need for filtering
3. **Namespace Design**: Use namespaces for logical separation
4. **Index Updates**: Use `update()` for metadata changes
5. **Connection Pooling**: Reuse connections in production
6. **Async Operations**: Use async for parallel processing

---

## 9. Cost Management

In [ ]:
# Monitor usage
def get_usage_stats():
    """Get index statistics for cost estimation"""
    stats = index.describe_index_stats()
    
    total_vectors = stats['total_vector_count']
    dimension = stats['dimension']
    
    # Estimate storage (rough)
    storage_mb = (total_vectors * dimension * 4) / (1024 * 1024)  # 4 bytes per float
    
    print("Usage Statistics:")
    print(f"  Total vectors: {total_vectors:,}")
    print(f"  Dimension: {dimension}")
    print(f"  Estimated storage: {storage_mb:.2f} MB")
    
    # Per namespace
    namespaces = stats.get('namespaces', {})
    print(f"\n  Namespaces:")
    for ns, ns_stats in namespaces.items():
        print(f"    {ns}: {ns_stats['vector_count']:,} vectors")
    
    return stats

usage = get_usage_stats()

# Cost optimization tips
print("\n💡 Cost Optimization Tips:")
print("  1. Delete unused namespaces")
print("  2. Use serverless for variable workloads")
print("  3. Minimize metadata size")
print("  4. Monitor query patterns")
print("  5. Use caching for frequent queries")

---

## 10. Production RAG System with Pinecone

In [ ]:
%%writefile pinecone_rag.py
from pinecone import Pinecone
from openai import OpenAI
import os

class ProductionRAG:
    def __init__(self, index_name: str):
        # Initialize clients
        self.pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
        self.openai_client = OpenAI()
        self.index = self.pc.Index(index_name)
    
    def embed(self, text: str) -> list:
        """Generate embedding"""
        response = self.openai_client.embeddings.create(
            model="text-embedding-ada-002",
            input=text
        )
        return response.data[0].embedding
    
    def add_documents(self, documents: list, namespace: str = "default"):
        """Add documents to index"""
        vectors = [
            {
                "id": doc["id"],
                "values": self.embed(doc["text"]),
                "metadata": {"text": doc["text"], **doc.get("metadata", {})}
            }
            for doc in documents
        ]
        
        # Batch upsert
        batch_size = 100
        for i in range(0, len(vectors), batch_size):
            batch = vectors[i:i+batch_size]
            self.index.upsert(vectors=batch, namespace=namespace)
    
    def retrieve(self, query: str, top_k: int = 5, 
                filter_dict: dict = None, namespace: str = "default"):
        """Retrieve relevant documents"""
        query_embedding = self.embed(query)
        
        results = self.index.query(
            vector=query_embedding,
            top_k=top_k,
            filter=filter_dict,
            namespace=namespace,
            include_metadata=True
        )
        
        return [
            {
                "id": match["id"],
                "score": match["score"],
                "text": match["metadata"]["text"]
            }
            for match in results["matches"]
        ]
    
    def generate(self, query: str, context: list, system_prompt: str = None):
        """Generate answer using retrieved context"""
        # Build context string
        context_str = "\n\n".join([doc["text"] for doc in context])
        
        # System prompt
        if system_prompt is None:
            system_prompt = "You are a helpful AI assistant. Answer based on the provided context."
        
        # Generate
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Context:\n{context_str}\n\nQuestion: {query}"}
        ]
        
        response = self.openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0.7
        )
        
        return response.choices[0].message.content
    
    def query(self, query: str, top_k: int = 5, 
             filter_dict: dict = None, namespace: str = "default"):
        """Complete RAG pipeline"""
        # Retrieve
        context = self.retrieve(query, top_k, filter_dict, namespace)
        
        # Generate
        answer = self.generate(query, context)
        
        return {
            "question": query,
            "answer": answer,
            "sources": context
        }

print("✅ Production RAG class created!")

# Usage:
# rag = ProductionRAG("my-index")
# response = rag.query("What is machine learning?")
# print(response["answer"])

---

## 🎯 Practice Exercises

### Exercise 1: Build Knowledge Base
Create a knowledge base with:
- 100+ documents
- Multiple categories
- Rich metadata
- Test search with filters

### Exercise 2: Multi-Tenant System
Build multi-tenant RAG:
- 3+ namespaces (tenants)
- Isolated data per tenant
- Tenant-specific search
- Usage analytics per tenant

### Exercise 3: Hybrid Search
Implement production hybrid search:
- Semantic + keyword
- Configurable weighting
- Result reranking
- A/B test different alphas

### Exercise 4: Performance Benchmark
Test Pinecone performance:
- Index build time
- Query latency
- Batch vs individual
- Different top_k values

### Challenge: Production System
Build complete system:
- Document ingestion pipeline
- Real-time updates
- Multi-namespace support
- Hybrid search
- Monitoring & alerts
- Cost tracking
- FastAPI endpoints

---

## 📚 Key Takeaways

1. **Pinecone** - Managed vector database, easy to use
2. **Indexes** - Container for vectors
3. **Namespaces** - Data isolation for multi-tenancy
4. **Metadata** - Essential for filtering
5. **Hybrid Search** - Combine semantic + keyword
6. **Batch Operations** - For performance
7. **Cost Management** - Monitor and optimize
8. **Production Ready** - Built for scale

---

## 🚀 Next Steps

**AP21 - Weaviate for Semantic Search**
- Open-source alternative
- GraphQL queries
- Generative search
- Multi-tenancy

**Resources:**
- [Pinecone Documentation](https://docs.pinecone.io/)
- [Pinecone Examples](https://github.com/pinecone-io/examples)
- [Pinecone Community](https://community.pinecone.io/)

---

*Created as part of the Applied AI module*  
*Last updated: January 2025*